# Data Check 
**qa/qc to double check data after sampling trips**
<br>
- Do sample health statuses match colony conditions?
    - *done*

- Check on 'dead' corals: did they stay dead (if not, change condition bc they were never dead),were they checked on, were any samples taken (possibly from the wrong coral?)
    - *have not started*

- Did we take samples from all live corals - 2 per coral, 3 on immune trips 
    - this is more complicated with health status 
    - *in progress*

- Summary stats:
    - Compare # of expected samples to how many samples were taken 
        - *done*
    - Num of corals at each transect (how does this compare to the previous trip?)
        - *done*
    - Num of corals within each condition
        - *done*
<br>
<br>
- *this script is in progress - last updated 4/29/2026*

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
os.getcwd()

'/Users/brookesienkiewicz/Documents/Code_notebook/QA_QC'

In [3]:
# upload sample data 
sample_data=pd.read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Sample_Data/CBC_samples.csv',index_col="Tubelabel_species")

In [4]:
# convert dates to str
sample_data['Month_year'] = sample_data['Month_year'].astype(str)
# remove the decimal point
sample_data['Month_year'] = sample_data['Month_year'].str.replace('.0', '')
# add leading zeros where necessary
sample_data['Month_year'] = sample_data['Month_year'].str.pad(width=6, side='left', fillchar='0')
sample_data['Month_year']

# Extract month and year to make separate columns 
sample_data.loc[:,'Month'] = sample_data.loc[:,'Month_year'].str[0:2]
sample_data.loc[:,'Year'] = sample_data.loc[:,'Month_year'].str[2:]

In [5]:
# upload colony data 
colony_data=pd.read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Sample_Data/CBC_ColonyData.csv')

In [6]:
# convert integers to strings 
colony_data['NewTagNum'] = colony_data['NewTagNum'].astype(str)
colony_data['TransectNum'] = colony_data['TransectNum'].astype(str)
sample_data['NewTagNum'] = sample_data['NewTagNum'].astype(str)
sample_data['TransectNum'] = sample_data['TransectNum'].astype(str)

In [7]:
# drop 'unnamed' col
colony_data = colony_data.drop(columns=['Unnamed: 0'])

In [8]:
# fix caps issues 
colony_data = colony_data.applymap(lambda x: 'Not_Visited' if isinstance(x, str) and x.lower() == 'not_visited' else x)
colony_data['062025_Condition'].unique()

/var/folders/_7/b8vq3n312mz355q4ddfk9h1w0000gq/T/ipykernel_19628/2697612767.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  colony_data = colony_data.applymap(lambda x: 'Not_Visited' if isinstance(x, str) and x.lower() == 'not_visited' else x)


array(['Diseased', 'Healthy', 'Dead', 'CLP', nan, 'Diseased_Other',
       'Not_Visited', 'Healthy '], dtype=object)

In [9]:
# fix syntax issues
condition_cols = [col for col in colony_data.columns if col.endswith('_Condition')]
for col in condition_cols:
    colony_data[col] = colony_data[col].str.strip()

In [10]:
# add tag id to both sample and colony data to match colonies
colony_data['ID'] = colony_data['Species'] + '_T' + colony_data['TransectNum'] + '_' + colony_data['NewTagNum']
sample_data['ID'] = sample_data['Species'] + '_T' + sample_data['TransectNum'] + '_' + sample_data['NewTagNum']

In [11]:
# make 'ID' the index for sample and colony data 
sample_data.set_index('ID',inplace=True)
colony_data.set_index('ID',inplace=True)

In [12]:
#drop any accidental samples
sample_data = sample_data[sample_data['NewTagNum'] != "AS"]

In [13]:
# filter sample data to just get our samples from each date
uml_sample = sample_data[
    (sample_data['Sample_type'] == "Core_EtOH") | 
    (sample_data['Sample_type'] == "Core_RNAlater")
]


In [14]:
# change sample 102019 sample month years to 062019 since they are all grouped together in the condition data
uml_sample.loc[uml_sample['Month_year']=="102019","Month_year"]="062019"

In [15]:
# create for loop, cycle thru each data and see if sample health statuses match colony conditions 
# filter for etoh and rna later or will it work without??
# get unique values for month year 
dates=uml_sample['Month_year'].unique().tolist()

In [16]:
# replace NAs with Not_Visited
condition_cols=colony_data.columns[colony_data.columns.str.contains("Condition")]
colony_data[condition_cols] = colony_data[condition_cols].fillna("Not_Visited")

In [17]:
# check unique values in conditions
for date in dates:
    print(date,colony_data[f'{date}_Condition'].unique())
colony_data[condition_cols].describe()

092023 ['Diseased' 'CLP,CLB' 'Not_Visited' 'Dead' 'CLP' 'Healthy' 'Diseased, CLB'
 'CLB' 'Diseased, CLP' 'CLP,DC' 'DC' 'CLB,CLP']
122022 ['Diseased' 'Healthy' 'Dead' 'DC' 'Not_Visited' 'CLB']
052022 ['Diseased' 'Healthy' 'DC' 'Dead' 'Not_Visited' 'CLB']
112023 ['Not_Visited' 'CLB' 'CLP' 'Dead' 'CLP,CLB' 'Healthy']
042024 ['Diseased' 'Not_Visited' 'Healthy' 'Dead' 'Diseased, CLP' 'CLP'
 'Diseased_Other' 'DC']
062019 ['Healthy' 'Not_Visited']
122023 ['Not_Visited' 'CLB' 'Healthy' 'Dead' 'CLB,CLP' 'CLP' 'CLP,CLB' 'DC']
012024 ['Diseased' 'CLP,CLB' 'Healthy' 'Not_Visited' 'Dead' 'CLB' 'CLP' 'CLP,DC'
 'Diseased_Other, CLP' 'DC']
022024 ['Not_Visited' 'Healthy' 'Diseased' 'Dead' 'CLB' 'CLP' 'Diseased_Other'
 'DC']
062024 ['Not_Visited' 'Healthy' 'Diseased' 'Dead' 'DC' 'Diseased_Other']
082024 ['Not_Visited' 'Healthy' 'Diseased' 'Dead' 'CLP' 'Diseased_Other' 'DC'
 'CLB']
062025 ['Diseased' 'Healthy' 'Dead' 'CLP' 'Not_Visited' 'Diseased_Other']


,062019_Condition,052022_Condition,122022_Condition,092023_Condition,112023_Condition,122023_Condition,012024_Condition,022024_Condition,042024_Condition,062024_Condition,082024_Condition,122024_Condition,062025_Condition
count,220,220,220,220,220,220,220,220,220,220,220,220,220
unique,2,6,6,12,6,8,10,8,8,6,8,11,6
top,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Healthy,Not_Visited,Not_Visited,Not_Visited,Healthy
freq,113,104,84,105,181,182,97,182,114,177,183,70,114


## Mismatching Health Statuses - Colony & Sample Data

In [18]:
# create dfs by coral id incl condition and sample health status at each time point 
dataframes_dict = {}
# run for loop
for date in dates:
    # get corresponding condition col in colony data for each date 
    condition_col = colony_data.loc[:, colony_data.columns.str.contains(f'{date}_Condition')]
    # convert data_condition col into df and store in dict 
    condition_df=pd.DataFrame(condition_col)
    dataframes_dict[date] = condition_df

    # make df containing samples at the same point, showing their sample type and health status
    sample_health = uml_sample[uml_sample["Month_year"] == date][["Sample_type", "Health_status"]].groupby(level=0).apply(
        lambda x: pd.Series({
            'Sample_type': ', '.join(x['Sample_type'].unique()),
            'Health_status': ', '.join(x['Health_status'].dropna().unique())
        })
    )
     
    # merge
    merged_all = pd.merge(
        condition_df, 
        sample_health, 
        left_index=True, 
        right_index=True, 
        how='left')
    dataframes_dict[f'{date}_all'] = merged_all

    # 
    condition_col_name = f'{date}_Condition'
    visited_mask = ~merged_all[condition_col_name].isin(["Not_Visited", "Dead"])
    dataframes_dict[f'{date}_visited'] = merged_all[visited_mask]


In [19]:
# each date has 3 dfs: 
    # date: one with all conditions,
    # date_all: one with all colonies (visited and not visited) + sample health statuses listed
    # date_visited: one with just visited colonies w/ conditions + sample health statuses listed


In [20]:
# also need to check if non-visited colonies are supposed to be visited (by checking previous health statuses)
#  also want to: sum up alive and visited corals and number of samples taken from each individual colony 
# should add all non-sampled into one df for easy viewing

In [21]:
# cycle thru samples dicts and view any mismatches 
# Initialize a list to store mismatches
mismatches = []

for date in dates:
    df = dataframes_dict[f'{date}_all']
    cond_col = f'{date}_Condition'

    # Condition 1: Dead/Not Visited
    cond1_mask = (df[cond_col].isin(["Not_Visited", "Dead"])) & (df['Health_status'].notna())
    if cond1_mask.any():
        mismatches.append((date, 'Condition 1 Violations', df[cond1_mask]))

    # Condition 2: Live colonies - health status matching
    # group by the index (colony ID) to see all samples for a colony at once
    c2_list = []
    for colony_id, group in df.groupby(level=0):
        cond = group[cond_col].iloc[0]
        
        # Skip Dead/Not_Visited/NaN (those are Condition 1 or missing data)
        if cond in ["Not_Visited", "Dead"] or pd.isna(cond):
            continue
            
        statuses = group['Health_status'].tolist()
        is_violation = False

        # A. Healthy: All samples must be 'Healthy'
        if cond == 'Healthy':
            if not all(s == 'Healthy' for s in statuses):
                is_violation = True

        # B. Diseased: Must have at least one 'Diseased_Tissue' AND one 'Diseased_Margin'
        elif 'Diseased' in str(cond):
            all_status_str = " ".join([str(s) for s in statuses])
            has_tissue = 'Diseased_Tissue' in all_status_str
            has_margin = 'Diseased_Margin' in all_status_str
            if not (has_tissue and has_margin):
                is_violation = True

        # C. Bleaching: If condition contains CLP/CLB, must have 'Bleached_Tissue'
        elif any(term in str(cond) for term in ['CLP', 'CLB']):
            if 'Bleached_Tissue' not in statuses:
                is_violation = True
        
        # D. Simple Mismatch: Catch-all for other conditions
        elif any(s != cond for s in statuses if pd.notna(s)):
            is_violation = True

        if is_violation:
            c2_list.append(group)

    if c2_list:
        mismatches.append((date, 'Condition 2 Violations', pd.concat(c2_list)))

# print results
if mismatches:
    for date, violation_type, df_mismatch in mismatches:
        print(f"Date: {date} - {violation_type}")
        # Reset index just for printing so 'ID' is visible
        print(df_mismatch.reset_index()[['ID', f'{date}_Condition', 'Health_status']].to_string(index=False))
        print("\n")
else:
    print("No mismatches found.")

Date: 092023 - Condition 2 Violations
        ID 092023_Condition                    Health_status
DLAB_T3_69              CLP                          Healthy
DLAB_T4_93          Healthy Diseased_Tissue, Diseased_Margin
OANN_T1_23              CLB Diseased_Tissue, Diseased_Margin
OANN_T2_30          CLP,CLB Diseased_Margin, Diseased_Tissue
OFAV_T2_80          CLP,CLB Diseased_Margin, Diseased_Tissue
OFAV_T4_77          Healthy Diseased_Tissue, Diseased_Margin
PAST_T1_19              CLB                              NaN
 SSID_T1_1         Diseased                  Diseased_Margin
SSID_T1_11         Diseased                  Diseased_Margin
 SSID_T1_3         Diseased                          Healthy
 SSID_T1_9         Diseased                  Diseased_Margin
SSID_T2_66              CLB                  Diseased_Margin
SSID_T2_67              CLB                  Diseased_Margin
SSID_T2_71              CLP                  Diseased_Margin
SSID_T2_74         Diseased         Healthy, Di

## Summary Stats

In [22]:
# Convert 'Date_InitialTag' to datetime format and then to 'mmyyyy' format
colony_data['Tag_Date_mmyyyy'] = pd.to_datetime(colony_data['Date_InitialTag']).dt.strftime('%m%Y')

/var/folders/_7/b8vq3n312mz355q4ddfk9h1w0000gq/T/ipykernel_19628/1664916380.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  colony_data['Tag_Date_mmyyyy'] = pd.to_datetime(colony_data['Date_InitialTag']).dt.strftime('%m%Y')


In [23]:
# exclude tem, na, or syringe for now 
filtered_samples = sample_data[
    (sample_data['Sample_type'] != 'TEM') & 
    (sample_data['Sample_type'] != 'Syringe') & 
    (sample_data['Sample_type'].notna())
]


In [24]:
# make all t names consistent 
filtered_samples.loc[(filtered_samples['Transect'].str.contains("Curlew",case=False)),'Transect']='CURLEW'
filtered_samples.loc[(filtered_samples['Transect'].str.contains("Lagoon",case=False)),'Transect']='LAGOON'
filtered_samples.loc[(filtered_samples['Transect'].str.contains("Hangman",case=False)),'Transect']='HANGMAN'
filtered_samples['Transect'].unique()

array(['CURLEW', 'CBC30N', 'SR30N', 'LAGOON', 'BB', 'HANGMAN'],
      dtype=object)

In [33]:
# make sorted list of transects 
all_transects = filtered_samples['Transect'].unique()
sorted_transects = sorted(all_transects)

In [29]:
#make monthyear in chronological order 
sorted_dates=['062019', '052022', '122022', '092023', '112023', '122023',
       '012024', '022024', '042024', '062024', '082024', '062025']
immune_dates=['112023', '122023', '012024', '022024','062024', '082024']

### Colony Numbers - All 
- number of tagged, live, and sampled colonies per date x transect

In [30]:
from pandas import NA


summary_data = []

for date in sorted_dates:
    date_samples = filtered_samples[filtered_samples['Month_year'] == date]
    
    for transect in sorted_transects:
        # filter colony and sample data
        transect_colony = colony_data[colony_data['Transect'] == transect]
        transect_samples = date_samples[date_samples['Transect'] == transect]

        # get numbers of tagged colonies, live colonies, and sampled colonies per transect
        total_tagged=len(transect_colony['NewTagNum'].unique())
        num_live = transect_colony[~transect_colony[f'{date}_Condition'].isin(["Not_Visited", "Dead", "Unknown","NaN"])].shape[0]
        if date in immune_dates:
            num_immune_tag = len(transect_colony[transect_colony[f'immune_y/n'] == 'y'])
        else:
            num_immune_tag = NA
        num_tag_sampled = len(transect_samples['NewTagNum'].unique())
        
        # sample counts are only helpful when we connect to health status

        # # Count samples
        # num_uml_samples = num_core_samples = transect_samples[
        #     (transect_samples['Sample_type'] == 'Core_EtOH') | 
        #     (transect_samples['Sample_type'] == 'Core_RNAlater')
        # ].shape[0]
        # num_frozen_samples = transect_samples[transect_samples['Sample_type'] == 'Core_frozen'].shape[0]
        # num_immune_samples = transect_samples[transect_samples['Sample_type'] == 'Immune'].shape[0]
        
        # Append the results to summary_data
        summary_data.append({
            'Date': date,
            'Transect': transect,
            'total_tagged': total_tagged,
            'num_live': num_live,
            'num_immune_tag': num_immune_tag,
            'num_tag_sampled': num_tag_sampled,
        })

# Convert summary_data into a DataFrame
summary_df = pd.DataFrame(summary_data)
# remove row if total_tagged is 0
summary_df = summary_df[summary_df['num_live'] > 0]

# Display the summary DataFrame
summary_df


,Date,Transect,total_tagged,num_live,num_immune_tag,num_tag_sampled
1,062019,CBC30N,36,31,<NA>,24
4,062019,LAGOON,50,39,<NA>,31
5,062019,SR30N,47,37,<NA>,28
7,052022,CBC30N,36,26,<NA>,26
10,052022,LAGOON,50,28,<NA>,24
11,052022,SR30N,47,30,<NA>,29
13,122022,CBC30N,36,23,<NA>,22
14,122022,CURLEW,17,17,<NA>,16
16,122022,LAGOON,50,35,<NA>,34
17,122022,SR30N,47,30,<NA>,31


### Colony Numbers - Previous Trip Summary
- filter tagged numbers for single trip
- add diseased colonies and sample numbers
- predict number of samples for next trip:

In [31]:
# calculate predicted sample numbers for next trip
previous_date='062025' #update previous trip date
trip_df = summary_df[summary_df['Date'] == previous_date]
trip_df=trip_df.drop('num_immune_tag', axis=1)

In [32]:
trip_df
# add diseased column
trip_df['num_diseased'] = 
    transect_colony[~transect_colony[f'{date}_Condition'].isin(["Not_Visited", "Dead", "Unknown","NaN"])].shape[0]

SyntaxError: invalid syntax (190151440.py, line 3)

In [ ]:
to_merge=[]
# can add for loop for dates if need to check total sample numbers
for transect in sorted_transects:
    transect_colony = colony_data[colony_data['Transect'] == transect]
    transect_samples = date_samples[date_samples['Transect'] == transect]

    num_tag_diseased = len(transect_colony[transect_colony[f'{previous_date}_Condition'] == 'Diseased'])

    num_uml_samples = transect_samples[
                (transect_samples['Sample_type'] == 'Core_EtOH') | 
                (transect_samples['Sample_type'] == 'Core_RNAlater')].shape[0]
    num_frozen_samples = transect_samples[transect_samples['Sample_type'] == 'Core_frozen'].shape[0]

    to_merge.append({
            'Date': previous_date,
            'Transect': transect,
            'num_tag_diseased': num_tag_diseased,
            'num_uml_samples': num_uml_samples, 
            'num_frozen_samples': num_frozen_samples,
        })
to_merge = pd.DataFrame(to_merge)
trip_df.merge(to_merge, on=['Date', 'Transect'], how='left')

,Date,Transect,total_tagged,num_live,num_tag_sampled,num_tag_diseased,num_uml_samples,num_frozen_samples
0,062025,BB,37,36,36,2,38,38
1,062025,CBC30N,36,15,15,7,22,22
2,062025,CURLEW,17,12,11,0,11,11
3,062025,HANGMAN,31,31,31,0,31,31
4,062025,LAGOON,50,25,25,6,30,29
5,062025,SR30N,47,26,25,4,30,30


In [ ]:
# predict number of samples for next trip
trip_df['predicted_samples'] = ((trip_df['num_live'] * 2) + # 2 samples per healthy colony
                                (trip_df['num_live'] * 0.25)).round(0).astype(int) # add extra in case of disease


In [ ]:
trip_df

,Date,Transect,total_tagged,num_live,num_tag_sampled,predicted_samples
66,062025,BB,37,36,36,81
67,062025,CBC30N,36,15,15,34
68,062025,CURLEW,17,12,11,27
69,062025,HANGMAN,31,31,31,70
70,062025,LAGOON,50,25,25,56
71,062025,SR30N,47,26,25,58


In [ ]:
print("Total number of live colonies for next trip:")
print(trip_df['num_live'].sum())

print("Predicted total number of samples for next trip:")
print(trip_df['predicted_samples'].sum())

Total number of live colonies for next trip:
145
Predicted total number of samples for next trip:
326


### Colony Numbers, Species, and Health Statuses
- per date x transect

In [ ]:
species_data = []

for date in sorted_dates:
    date_samples = filtered_samples[filtered_samples['Month_year'] == date]
    
    for transect in sorted_transects:
        # filter colony and sample data
        transect_colony = colony_data[colony_data['Transect'] == transect]
        transect_samples = date_samples[date_samples['Transect'] == transect]

        # get numbers of tagged colonies, live colonies, and sampled colonies per transect
        total_tagged=len(transect_colony['NewTagNum'].unique())
        num_live = transect_colony[~transect_colony[f'{date}_Condition'].isin(["Not_Visited", "Dead", "Unknown","NaN"])].shape[0]

        for species in colony_data['Species'].unique():
            species_colony = transect_colony[transect_colony['Species'] == species]
            species_samples = transect_samples[transect_samples['Species'] == species]

            species_tagged_total = len(species_colony['NewTagNum'].unique())
            species_live = species_colony[~species_colony[f'{date}_Condition'].isin(["Not_Visited", "Dead", "Unknown","NaN"])].shape[0]
            species_healthy = species_colony[species_colony[f'{date}_Condition'].isin(["Healthy"])].shape[0]
            species_diseased = species_colony[species_colony[f'{date}_Condition'].isin(["Diseased"])].shape[0]
            species_clb = species_colony[species_colony[f'{date}_Condition'].isin(["CLP", "CLB"])].shape[0]

        # sample counts are only helpful when we connect to health status

        # # Count samples
        # num_uml_samples = num_core_samples = transect_samples[
        #     (transect_samples['Sample_type'] == 'Core_EtOH') | 
        #     (transect_samples['Sample_type'] == 'Core_RNAlater')
        # ].shape[0]
        # num_frozen_samples = transect_samples[transect_samples['Sample_type'] == 'Core_frozen'].shape[0]
        # num_immune_samples = transect_samples[transect_samples['Sample_type'] == 'Immune'].shape[0]
        
            # Append the results to summary_data
            species_data.append({
                'Date': date,
                'Transect': transect,
                'total_tagged': total_tagged,
                'total_live': num_live,
                'species': species,
                'species_tagged_total': species_tagged_total,
                'species_live': species_live,
                'species_healthy': species_healthy,
                'species_diseased': species_diseased,
                'species_clb': species_clb,
            })

# Convert summary_data into a DataFrame
species_data = pd.DataFrame(species_data)
# remove row if total_tagged is 0
species_data = species_data[species_data['total_live'] > 0]

# Display the summary DataFrame
# species_data.pivot(index=['Date', 'species'], columns='Transect', values=['species_tagged_total', 'species_live', 'species_healthy', 'species_diseased', 'species_clb'])
species_data


,Date,Transect,total_tagged,total_live,species,species_tagged_total,species_live,species_healthy,species_diseased,species_clb
9,062019,CBC30N,36,31,SSID,8,8,8,0,0
10,062019,CBC30N,36,31,PAST,5,5,5,0,0
11,062019,CBC30N,36,31,PSTR,7,7,7,0,0
12,062019,CBC30N,36,31,MCAV,9,9,9,0,0
13,062019,CBC30N,36,31,OANN,3,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
643,062025,SR30N,47,26,OANN,2,2,1,0,1
644,062025,SR30N,47,26,DLAB,2,2,2,0,0
645,062025,SR30N,47,26,MMEA,6,0,0,0,0
646,062025,SR30N,47,26,OFAV,4,4,4,0,0


In [ ]:
#prep
value_vars = ['species_live', 'species_healthy', 'species_diseased', 'species_clb']
df_melted = species_data.melt(
    id_vars=['Date', 'Transect', 'species'], 
    value_vars=value_vars, 
    var_name='Metric', 
    value_name='Count'
)
df_melted['Count'] = pd.to_numeric(df_melted['Count'], errors='coerce')
# pivot
hierarchical_pivot = df_melted.pivot_table(
    index=['Transect', 'species', 'Metric'], 
    columns='Date', 
    values='Count', 
    aggfunc='first'
)
# filter and clean
new_metrics = [n.replace('species_', '').title() for n in hierarchical_pivot.index.levels[2]]
hierarchical_pivot.index = hierarchical_pivot.index.set_levels(new_metrics, level=2)
hierarchical_pivot.index.names = ['Transect', 'Species', 'Metric']
is_live_row = hierarchical_pivot.index.get_level_values('Metric') == 'Live'
has_data = (hierarchical_pivot.fillna(0) > 0).any(axis=1)
hierarchical_pivot = hierarchical_pivot[is_live_row | has_data]
hierarchical_pivot = hierarchical_pivot.astype('Int64')
hierarchical_pivot = hierarchical_pivot.astype(object)
final_table = hierarchical_pivot.fillna('-')
final_table.columns = pd.to_datetime(final_table.columns, format='%m%Y')

# 2. Sort the columns chronologically
final_table = final_table.sort_index(axis=1)
final_table.columns = final_table.columns.strftime('%m%Y')

/var/folders/_7/b8vq3n312mz355q4ddfk9h1w0000gq/T/ipykernel_7089/1525231194.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_table = hierarchical_pivot.fillna('-')


In [ ]:
final_table

Date                      062019 052022 122022 092023 112023 122023 012024  \
Transect Species Metric                                                      
BB       CNAT    Live          -      -      -      -      -      -      0   
         DLAB    Live          -      -      -      -      -      -      0   
         MCAV    Clb           -      -      -      -      -      -      1   
                 Healthy       -      -      -      -      -      -      2   
                 Live          -      -      -      -      -      -      6   
...                          ...    ...    ...    ...    ...    ...    ...   
SR30N    PSTR    Live          7      5      4      2      0      0      1   
         SSID    Clb           0      1      2      5      2      3      4   
                 Diseased      0      6      6      3      0      0      1   
                 Healthy      10      2      0      0      0      0      0   
                 Live         10      9      8      9      3      3      5   

Date                      022024  042024 062024 082024  062025  
Transect Species Metric                                         
BB       CNAT    Live          -       0      -      -       0  
         DLAB    Live          -       0      -      -       0  
         MCAV    Clb           -       0      -      -       0  
                 Healthy       -       2      -      -       6  
                 Live          -       5      -      -       6  
...                          ...     ...    ...    ...     ...  
SR30N    PSTR    Live          0       2      0      0       2  
         SSID    Clb           2       0      0      3       0  
                 Diseased      1       6      3      0       4  
                 Healthy       0       1      1      0       3  
                 Live          3       7      5      3       7  

[143 rows x 12 columns]

In [ ]:
# all current live colonies
# from os import write

current_df=final_table.xs('Live', level='Metric')[[previous_date]]
transect_columns = current_df.unstack(level=0)
transect_columns.columns = transect_columns.columns.get_level_values(1)
transect_columns.index.name = 'Species'
transect_columns['Total'] = transect_columns.sum(axis=1)
transect_columns.loc['Total'] = transect_columns.sum(axis=0)
transect_columns
transect_columns.to_csv('/Users/brookesienkiewicz/Documents/Code_notebook/Data_Summaries/current_live_colonies.csv')

In [ ]:
# view all with health statuses
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    # print(final_table[[previous_date]])

IndentationError: unexpected indent (2433903902.py, line 2)